In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    FloatSlider,
    IntSlider,
    Dropdown,
    interactive,
    VBox,
    HBox,
    Layout,
    HTMLMath,
    HTML
)

from IPython.display import display, Markdown

# ============================================================
# RANDOM SEQUENCES — MEAN AND AUTOCORRELATION
# ============================================================

display(Markdown(r"""
## Random Sequences, Stationarity and Autocorrelation

A random sequence is a collection of random variables indexed by
discrete time. Its statistical behavior can be described by quantities
such as the mean and the autocorrelation function.

For a wide-sense stationary (WSS) random sequence, the mean is constant
and the autocorrelation depends only on the time lag rather than on the
two time indices separately.
"""))

# ============================================================
# CONTROLS
# ============================================================

process_dropdown = Dropdown(
    options=[
        'White noise',
        'AR(1)'
    ],
    value='AR(1)',
    description='Process:',
    layout=Layout(width='300px')
)

a_slider = FloatSlider(
    value=0.80,
    min=-0.90,
    max=0.90,
    step=0.05,
    description='a:',
    continuous_update=True,
    layout=Layout(width='300px')
)

sigma_w_slider = FloatSlider(
    value=1.0,
    min=0.2,
    max=2.0,
    step=0.1,
    description='σw:',
    continuous_update=True,
    layout=Layout(width='300px')
)

N_slider = IntSlider(
    value=200,
    min=60,
    max=500,
    step=20,
    description='Length N:',
    continuous_update=True,
    layout=Layout(width='300px')
)

realizations_slider = IntSlider(
    value=300,
    min=50,
    max=1000,
    step=50,
    description='Realizations:',
    continuous_update=True,
    layout=Layout(width='300px')
)

# ============================================================
# CUSTOM CONTROL LAYOUT
# ============================================================

left_controls = VBox(
    [
        a_slider,
        sigma_w_slider
    ],
    layout=Layout(
        width='320px'
    )
)

right_controls = VBox(
    [
        N_slider,
        realizations_slider
    ],
    layout=Layout(
        width='320px'
    )
)

slider_columns = HBox(
    [
        left_controls,
        right_controls
    ],
    layout=Layout(
        width='680px',
        gap='25px',
        align_items='flex-start'
    )
)

controls = VBox(
    [
        process_dropdown,
        slider_columns
    ],
    layout=Layout(
        width='700px'
    )
)

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def random_sequence_demo(
    process_type,
    a,
    sigma_w,
    N,
    realizations
):

    rng = np.random.default_rng(12345)

    # --------------------------------------------------------
    # Generate ensemble
    # --------------------------------------------------------

    X = np.zeros(
        (
            realizations,
            N
        )
    )

    # ========================================================
    # WHITE NOISE
    # ========================================================

    if process_type == 'White noise':

        X = rng.normal(
            0.0,
            sigma_w,
            size=(
                realizations,
                N
            )
        )

    # ========================================================
    # AR(1)
    #
    # X[n] = a X[n-1] + W[n]
    # ========================================================

    elif process_type == 'AR(1)':

        stationary_sigma = (
            sigma_w
            /
            np.sqrt(
                1 - a**2
            )
        )

        X[:, 0] = rng.normal(
            0.0,
            stationary_sigma,
            size=realizations
        )

        W = rng.normal(
            0.0,
            sigma_w,
            size=(
                realizations,
                N
            )
        )

        for n in range(1, N):

            X[:, n] = (
                a * X[:, n - 1]
                +
                W[:, n]
            )

    # ========================================================
    # ENSEMBLE MEAN
    # ========================================================

    ensemble_mean = np.mean(
        X,
        axis=0
    )

    # ========================================================
    # ESTIMATED AUTOCORRELATION
    # ========================================================

    max_lag = min(
        40,
        N // 3
    )

    lags = np.arange(
        0,
        max_lag + 1
    )

    R_est = np.zeros(
        len(lags)
    )

    for i, lag in enumerate(lags):

        if lag == 0:

            products = (
                X * X
            )

        else:

            products = (
                X[:, :-lag]
                *
                X[:, lag:]
            )

        R_est[i] = np.mean(
            products
        )

    # ========================================================
    # THEORETICAL AUTOCORRELATION
    # ========================================================

    if process_type == 'White noise':

        R_theory = np.zeros_like(
            lags,
            dtype=float
        )

        R_theory[0] = (
            sigma_w**2
        )

        theoretical_variance = (
            sigma_w**2
        )

    else:

        theoretical_variance = (
            sigma_w**2
            /
            (1 - a**2)
        )

        R_theory = (
            theoretical_variance
            *
            a**lags
        )

    # ========================================================
    # FIGURES
    # ========================================================

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(12, 7)
    )

    n_axis = np.arange(N)

    # --------------------------------------------------------
    # 1. SAMPLE REALIZATIONS
    # --------------------------------------------------------

    number_to_show = min(
        5,
        realizations
    )

    for r in range(number_to_show):

        axes[0, 0].plot(
            n_axis,
            X[r, :],
            linewidth=1.0,
            alpha=0.75
        )

    axes[0, 0].set_title(
        'Sample Realizations'
    )

    axes[0, 0].set_xlabel('n')
    axes[0, 0].set_ylabel('X[n]')

    axes[0, 0].set_xlim(
        0,
        N - 1
    )

    axes[0, 0].set_ylim(
        -10,
        10
    )

    axes[0, 0].grid(
        True,
        alpha=0.25
    )

    # --------------------------------------------------------
    # 2. ENSEMBLE MEAN
    # --------------------------------------------------------

    axes[0, 1].plot(
        n_axis,
        ensemble_mean,
        linewidth=2
    )

    axes[0, 1].axhline(
        0,
        linestyle='--',
        linewidth=1.3
    )

    axes[0, 1].set_title(
        'Estimated Ensemble Mean'
    )

    axes[0, 1].set_xlabel('n')
    axes[0, 1].set_ylabel('E{X[n]}')

    axes[0, 1].set_xlim(
        0,
        N - 1
    )

    axes[0, 1].set_ylim(
        -1.5,
        1.5
    )

    axes[0, 1].grid(
        True,
        alpha=0.25
    )

    # --------------------------------------------------------
    # 3. ESTIMATED AUTOCORRELATION
    # --------------------------------------------------------

    axes[1, 0].stem(
        lags,
        R_est
    )

    axes[1, 0].set_title(
        'Estimated Autocorrelation'
    )

    axes[1, 0].set_xlabel(
        'Lag k'
    )

    axes[1, 0].set_ylabel(
        'RXX[k]'
    )

    axes[1, 0].set_xlim(
        -1,
        max_lag + 1
    )

    axes[1, 0].set_ylim(
        -1,
        10
    )

    axes[1, 0].grid(
        True,
        alpha=0.25
    )

    # --------------------------------------------------------
    # 4. THEORY VS ESTIMATE
    # --------------------------------------------------------

    axes[1, 1].plot(
        lags,
        R_theory,
        linewidth=2.5,
        label='Theoretical'
    )

    axes[1, 1].plot(
        lags,
        R_est,
        linestyle='--',
        linewidth=1.8,
        marker='o',
        markersize=4,
        label='Estimated'
    )

    axes[1, 1].set_title(
        'Autocorrelation — Theory vs Estimate'
    )

    axes[1, 1].set_xlabel(
        'Lag k'
    )

    axes[1, 1].set_ylabel(
        'RXX[k]'
    )

    axes[1, 1].set_xlim(
        0,
        max_lag
    )

    axes[1, 1].set_ylim(
        -1,
        10
    )

    axes[1, 1].grid(
        True,
        alpha=0.25
    )

    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # ========================================================
    # RESULTS — ALL ON ONE LINE
    # ========================================================

    mean_output = HTMLMath(
        value=(
            r'\('
            r'\widehat{\mu}_X='
            + f'{np.mean(X):.6f}'
            + r'\)'
        ),
        layout=Layout(width='165px')
    )

    R0_output = HTMLMath(
        value=(
            r'\('
            r'\widehat{R}_{XX}[0]='
            + f'{R_est[0]:.6f}'
            + r'\)'
        ),
        layout=Layout(width='190px')
    )

    variance_output = HTMLMath(
        value=(
            r'\('
            r'\operatorname{Var}(X)='
            + f'{theoretical_variance:.6f}'
            + r'\)'
        ),
        layout=Layout(width='190px')
    )

    # --------------------------------------------------------
    # Process-specific equations
    # --------------------------------------------------------

    if process_type == 'White noise':

        equation_output = HTMLMath(
            value=(
                r'\('
                r'R_{XX}[k]='
                + f'{sigma_w**2:.4f}'
                + r'\delta[k]'
                + r'\)'
            ),
            layout=Layout(width='220px')
        )

        equation_widgets = [
            equation_output
        ]

    else:

        process_output = HTMLMath(
            value=(
                r'\('
                r'X[n]='
                + f'{a:.2f}'
                + r'X[n-1]+W[n]'
                + r'\)'
            ),
            layout=Layout(width='205px')
        )

        correlation_output = HTMLMath(
            value=(
                r'\('
                r'R_{XX}[k]='
                r'\frac{\sigma_W^2}{1-a^2}'
                r'a^{|k|}'
                r'\)'
            ),
            layout=Layout(width='230px')
        )

        equation_widgets = [
            process_output,
            correlation_output
        ]

    results_row = HBox(
        [
            mean_output,
            R0_output,
            variance_output,
            *equation_widgets
        ],
        layout=Layout(
            width='1050px',
            gap='12px',
            align_items='center',
            justify_content='flex-start'
        )
    )

    display(results_row)

    # ========================================================
    # INTERPRETATION
    # ========================================================

    interpretation = HTML(
        value="""
        <div style="
            width: 900px;
            margin-top: 6px;
            font-family: Arial, sans-serif;
            font-size: 14px;
            line-height: 1.45;
        ">
        <b>Interpretation.</b><br>
        For both examples the ensemble mean is approximately constant.
        The autocorrelation depends only on the lag k, which is the
        characteristic second-order property of a wide-sense stationary
        random sequence. White noise is uncorrelated for every non-zero
        lag, whereas the AR(1) process exhibits correlation that gradually
        decays as the lag increases.
        </div>
        """
    )

    display(interpretation)


# ============================================================
# INTERACTIVE OBJECT
#
# We use interactive() because this architecture has proved
# reliable with %matplotlib inline.
# ============================================================

interactive_widget = interactive(
    random_sequence_demo,

    process_type=process_dropdown,
    a=a_slider,
    sigma_w=sigma_w_slider,
    N=N_slider,
    realizations=realizations_slider
)

# The last child of interactive() is its Output widget.
plot_output = interactive_widget.children[-1]

# ============================================================
# DISPLAY CUSTOM CONTROLS + INTERACTIVE OUTPUT
# ============================================================

display(
    controls,
    plot_output
)